# 🧠 WellSense — Student Mental Health Prediction
### University ML Assessment Project

**Project:** WellSense  
**Objective:** Predict student mental health risk (depression/anxiety) using classical ML models  
**Dataset:** Student Mental Health — Shariful07 (Kaggle)  
**Models Covered:** Logistic Regression, Decision Tree, Random Forest, SVM, k-NN, Naive Bayes, MLP  
**Syllabus Units:** II, III, IV, V  

---

## 📦 Phase 1 — Setup & Imports

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Imbalanced data handling
from imblearn.over_sampling import SMOTE

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

print('✅ All libraries imported successfully!')

---
## 📂 Phase 1 — Load Dataset

> **Download from Kaggle:** https://www.kaggle.com/datasets/shariful07/student-mental-health  
> Save the CSV file as `Student Mental health.csv` inside the `data/` folder.

In [ ]:
# Load dataset
df = pd.read_csv('../data/Student Mental health.csv')

print(f'✅ Dataset loaded!')
print(f'📐 Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
# Column names and types
print('📋 Columns:')
for col in df.columns:
    print(f'  {col} — {df[col].dtype}')

In [ ]:
# Rename columns for easier use
df.columns = [
    'timestamp', 'gender', 'age', 'course', 'year',
    'cgpa', 'marital_status', 'depression', 'anxiety',
    'panic_attack', 'treatment'
]

print('✅ Columns renamed!')
df.head()

In [ ]:
# Basic statistics
df.describe(include='all')

---
## 🔍 Phase 1 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Missing values check ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found!')
else:
    print('⚠️ Missing values detected:')
    print(missing_df)

In [ ]:
# ── Target variable distributions ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [['#2ecc71','#e74c3c'], ['#3498db','#e67e22'], ['#9b59b6','#1abc9c']]
targets = ['depression', 'anxiety', 'panic_attack']
titles  = ['Depression', 'Anxiety', 'Panic Attack']

for ax, col, title, clr in zip(axes, targets, titles, colors):
    df[col].value_counts().plot(kind='bar', ax=ax, color=clr)
    ax.set_title(f'{title} Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(rotation=0)
    for p in ax.patches:
        ax.annotate(str(p.get_height()), (p.get_x()+0.1, p.get_height()+0.3))

plt.suptitle('Target Variable Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Demographics ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['gender'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
    colors=['#74b9ff','#fd79a8','#55efc4'], startangle=90)
axes[0].set_title('Gender Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('')

df['year'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#6c5ce7')
axes[1].set_title('Year of Study', fontsize=13, fontweight='bold')
axes[1].tick_params(rotation=0)

df['cgpa'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='#00b894')
axes[2].set_title('CGPA Distribution', fontsize=13, fontweight='bold')
axes[2].tick_params(rotation=45)

plt.tight_layout()
plt.savefig('../assets/demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Depression by Gender & Year ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pd.crosstab(df['gender'], df['depression']).plot(kind='bar', ax=axes[0],
    color=['#2ecc71','#e74c3c'])
axes[0].set_title('Depression by Gender', fontsize=13, fontweight='bold')
axes[0].tick_params(rotation=0)
axes[0].legend(title='Depression')

pd.crosstab(df['year'], df['depression']).plot(kind='bar', ax=axes[1],
    color=['#3498db','#e74c3c'])
axes[1].set_title('Depression by Year of Study', fontsize=13, fontweight='bold')
axes[1].tick_params(rotation=0)
axes[1].legend(title='Depression')

plt.tight_layout()
plt.savefig('../assets/depression_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Age distribution ──────────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
sns.histplot(df['age'], bins=15, kde=True, color='#6c5ce7')
plt.title('Age Distribution of Students', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.tight_layout()
plt.savefig('../assets/age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Co-occurrence of mental health conditions ─────────────────────────────────
mental_health = df[['depression','anxiety','panic_attack']].copy()
mental_health = mental_health.replace({'Yes':1, 'No':0})

plt.figure(figsize=(7, 5))
sns.heatmap(mental_health.corr(), annot=True, cmap='RdYlGn', fmt='.2f',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Between Mental Health Conditions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ⚙️ Phase 2 — Preprocessing

In [ ]:
# ── Drop timestamp (not useful) ───────────────────────────────────────────────
df_clean = df.drop(columns=['timestamp'])

# ── Handle missing values ─────────────────────────────────────────────────────
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

df_clean['age'].fillna(df_clean['age'].median(), inplace=True)

print('✅ Missing values handled!')
print(df_clean.isnull().sum())

In [ ]:
# ── Label Encoding ────────────────────────────────────────────────────────────
le = LabelEncoder()
categorical_cols = ['gender', 'course', 'year', 'cgpa', 'marital_status',
                    'depression', 'anxiety', 'panic_attack', 'treatment']

label_encoders = {}
for col in categorical_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

print('✅ Label encoding complete!')
df_clean.head()

In [ ]:
# ── Feature / Target split ────────────────────────────────────────────────────
# Target: depression (main prediction)
TARGET = 'depression'
FEATURES = [col for col in df_clean.columns if col != TARGET]

X = df_clean[FEATURES]
y = df_clean[TARGET]

print(f'Features: {FEATURES}')
print(f'Target  : {TARGET}')
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Class distribution:\n{y.value_counts()}')

In [ ]:
# ── Handle class imbalance with SMOTE ─────────────────────────────────────────
smote = SMOTE(random_state=RANDOM_STATE)
X_balanced, y_balanced = smote.fit_resample(X, y)

print(f'Before SMOTE: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'After  SMOTE: {dict(zip(*np.unique(y_balanced, return_counts=True)))}')

In [ ]:
# ── Feature Scaling ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

print('✅ Feature scaling done!')

In [ ]:
# ── Train / Test Split (80/20) ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_balanced, test_size=0.2, random_state=RANDOM_STATE, stratify=y_balanced
)

print(f'Training set  : {X_train.shape[0]} samples')
print(f'Test set      : {X_test.shape[0]} samples')

In [ ]:
# ── Optional: PCA for dimensionality reduction ────────────────────────────────
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)  # Keep 95% variance
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f'Original features    : {X_train.shape[1]}')
print(f'PCA components kept  : {X_train_pca.shape[1]}')
print(f'Variance explained   : {sum(pca.explained_variance_ratio_):.2%}')

# Plot explained variance
plt.figure(figsize=(9, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', color='#6c5ce7')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Explained Variance', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../assets/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🤖 Phase 3 — Model Building & Training

In [ ]:
# ── Define all models ─────────────────────────────────────────────────────────
models = {
    'Logistic Regression' : LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Decision Tree'       : DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest'       : RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    'SVM'                 : SVC(probability=True, random_state=RANDOM_STATE),
    'k-NN'                : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'         : GaussianNB(),
    'MLP (Neural Net)'    : MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500,
                                          random_state=RANDOM_STATE)
}

print(f'✅ {len(models)} models ready for training!')

In [ ]:
# ── Train all models & collect metrics ───────────────────────────────────────
results = {}
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    auc  = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    results[name] = {
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
        'ROC-AUC': round(auc, 4) if auc else 'N/A'
    }
    trained_models[name] = model
    print(f'Accuracy={acc:.4f} ✅')

print('\n🎉 All models trained!')

---
## 📊 Phase 4 — Evaluation & Comparison

In [ ]:
# ── Model comparison table ────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.sort_values('F1 Score', ascending=False)
results_df.index.name = 'Model'
print('📋 Model Comparison Table:')
results_df

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
plot_df = results_df[metrics].astype(float)

ax = plot_df.plot(kind='bar', figsize=(14, 6), width=0.75,
                  colormap='Set2', edgecolor='white')
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.tick_params(rotation=30)
plt.tight_layout()
plt.savefig('../assets/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices for all models ────────────────────────────────────────
n = len(models)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                linewidths=0.5)
    axes[idx].set_title(name, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for i in range(idx+1, len(axes)):
    fig.delaxes(axes[i])

plt.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../assets/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC-AUC Curves ────────────────────────────────────────────────────────────
plt.figure(figsize=(11, 7))
colors_roc = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22']

for (name, model), color in zip(trained_models.items(), colors_roc):
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

plt.plot([0,1],[0,1],'k--', label='Random Classifier', lw=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC-AUC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('../assets/roc_auc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Best model classification report ─────────────────────────────────────────
best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)

print(f'🏆 Best Model: {best_model_name}')
print('='*50)
print(classification_report(y_test, y_pred_best, target_names=['No Depression','Depression']))

In [ ]:
# ── Feature importance (Random Forest) ───────────────────────────────────────
rf_model = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#6c5ce7')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔧 Phase 5 — Optimization

In [ ]:
# ── K-Fold Cross Validation ───────────────────────────────────────────────────
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y_balanced, cv=kfold, scoring='f1_weighted')
    cv_results[name] = {'Mean F1': round(scores.mean(), 4), 'Std': round(scores.std(), 4)}
    print(f'{name:25s} → Mean F1: {scores.mean():.4f} ± {scores.std():.4f}')

cv_df = pd.DataFrame(cv_results).T.sort_values('Mean F1', ascending=False)
print('\n📋 Cross-Validation Results:')
cv_df

In [ ]:
# ── GridSearchCV — Hyperparameter Tuning (Random Forest) ─────────────────────
param_grid = {
    'n_estimators'  : [50, 100, 200],
    'max_depth'     : [None, 5, 10],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\n✅ Best Parameters : {grid_search.best_params_}')
print(f'✅ Best F1 Score   : {grid_search.best_score_:.4f}')

In [ ]:
# ── Evaluate tuned model ──────────────────────────────────────────────────────
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

print('🏆 Tuned Random Forest Results:')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'  F1 Score  : {f1_score(y_test, y_pred_tuned, average="weighted"):.4f}')
print(f'  ROC-AUC   : {roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1]):.4f}')

---
## 💾 Phase 6 — Save Models

In [ ]:
# ── Save best model, scaler, and encoders ─────────────────────────────────────
joblib.dump(best_rf,           '../models/best_model.pkl')
joblib.dump(scaler,            '../models/scaler.pkl')
joblib.dump(label_encoders,    '../models/label_encoders.pkl')
joblib.dump(results_df,        '../models/results_df.pkl')

print('✅ Model and artifacts saved to /models/ folder!')
print('   best_model.pkl')
print('   scaler.pkl')
print('   label_encoders.pkl')
print('   results_df.pkl')

---
## ✅ Summary

| Step | Description |
|------|-------------|
| Dataset | Student Mental Health (Kaggle - Shariful07) |
| Target | Depression prediction (binary classification) |
| Features | Gender, Age, Course, Year, CGPA, Marital Status, Anxiety, Panic Attack, Treatment |
| Preprocessing | Label encoding, StandardScaler, SMOTE, PCA |
| Models | LR, DT, RF, SVM, k-NN, Naive Bayes, MLP |
| Optimization | GridSearchCV, K-Fold Cross Validation |
| Best Model | See results table above |

> **Next Step:** Run `streamlit run app.py` from the project root to launch the WellSense web app.